# Board evaluation

In [127]:
import torch
import pandas as pd
import os
import chess

DATASET_DIR = os.path.join("..", "dataset")
MAIN_DATASET = os.path.join(DATASET_DIR, "chessData.csv")
df = pd.read_csv(MAIN_DATASET)

df["Evaluation"] = pd.to_numeric(df["Evaluation"], downcast="integer", errors="coerce")

In [128]:
def get_en_passant(fen):
    matrix = torch.zeros((8,8), dtype=torch.float64)
    fen_en_passant = fen.split(" ")[3]
    if fen_en_passant != "-":
        square = chess.parse_square(fen_en_passant)
        matrix[square % 8][square % 8] = 1
    return matrix

def get_piece_matrix(board, piece_type, color):
    matrix = torch.zeros((8,8), dtype=torch.float64)
    for sq in board.pieces(piece_type, color):
        r = 7 - chess.square_rank(sq)
        c = chess.square_file(sq)
        matrix[r, c] = 1
    return matrix

def fen_to_tensor(fen):
    board = chess.Board(fen)
    piece_index = {
        chess.PAWN : 1,
        chess.ROOK : 2,
        chess.KNIGHT : 3,
        chess.BISHOP : 4,
        chess.QUEEN : 5,
        chess.KING : 6,
    }
    fen_tensor = torch.zeros((13,8,8), dtype=torch.float64)
    fen_tensor[0] = get_en_passant(fen)
    for piece, val in piece_index.items():
        fen_tensor[val] = get_piece_matrix(board, piece, chess.BLACK)
        fen_tensor[val + 6] = get_piece_matrix(board, piece, chess.WHITE)
    return fen_tensor

In [129]:
for i in range(len(df)):
    fen_tensor = fen_to_tensor(df.iloc[i]["FEN"])
    eval_tensor = torch.tensor(df.iloc[i]["Evaluation"])
    
    print(fen_tensor)
    print(eval_tensor)
    break

tensor([[[0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.]],

        [[0., 0., 0., 0., 0., 0., 0., 0.],
         [1., 1., 1., 1., 1., 1., 1., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.]],

        [[1., 0., 0., 0., 0., 0., 0., 1.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
         [0., 0., 0., 0., 0., 0., 0., 0.],
       